# 06. Análise de Redes Complexas e Visualização de Grafos (Camada Ouro)

**Objetivo do Notebook:**
Consumir as tabelas colunares consolidadas da Camada Ouro (`gold_vertices` e `gold_arestas`) geradas pela Arquitetura Medallion para:
1. **Projetar a Rede de Coautoria**: Transformar o grafo bipartido (*Autor -> Trabalho*) em uma rede direta de colaboração científica ponderada (*Autor <-> Autor*).
2. **Extrair Métricas de Teoria dos Grafos**: Calcular graus de centralidade (*Degree Centrality*) e identificar os pesquisadores mais influentes (*hubs* de Inteligência Artificial).
3. **Detecção de Comunidades (Clustering)**: Identificar automaticamente grupos de pesquisa que publicam juntos utilizando o algoritmo de modularidade (*Greedy Modularity*).
4. **Renderização Interativa de Grafos na Web**: Gerar e renderizar um grafo com física de forças (*Force-Directed Layout*) diretamente no Databricks via `displayHTML()`.
5. **Exportação Científica (Gephi)**: Salvar a rede modelada no formato `.gexf` no Volume do Unity Catalog para permitir análises externas e reprodução científica.

In [ ]:
# Instalação da biblioteca NetworkX no cluster Databricks (Notebook-scoped)
%pip install networkx


In [ ]:
from pyspark.sql.functions import col, count, lit
import networkx as nx
from networkx.algorithms import community
import json

print("Carregando tabelas da Camada Ouro do Data Lakehouse...")
df_vertices = spark.table("gold_vertices")
df_arestas = spark.table("gold_arestas")

print(f"Total de Vértices: {df_vertices.count()}")
print(f"Total de Arestas: {df_arestas.count()}")


## 1. Projeção Relacional da Rede de Coautoria (Autor <-> Autor)
Na Camada Ouro, os relacionamentos são estruturados por arestas do tipo `Autoria` e `Citação`.
Para mapear como os pesquisadores colaboram entre si, executamos um auto-join otimizado sobre as arestas de autoria no Spark SQL, calculando a frequência de publicações conjuntas (`peso_coautoria`).

In [ ]:
# 1. Filtra as arestas de Autoria (Autor -> Trabalho)
df_autorias = df_arestas.filter(col("tipo_aresta") == "Autoria") \
    .select(
        col("id_origem").alias("autor_id"),
        col("id_destino").alias("work_id")
    )

# 2. Auto-join: se dois autores diferentes compartilham a mesma obra (work_id), há coautoria
df_coautorias = df_autorias.alias("a1") \
    .join(
        df_autorias.alias("a2"),
        (col("a1.work_id") == col("a2.work_id")) & (col("a1.autor_id") < col("a2.autor_id")),
        "inner"
    ) \
    .groupBy(
        col("a1.autor_id").alias("autor_1"),
        col("a2.autor_id").alias("autor_2")
    ) \
    .agg(count("*").alias("peso_coautoria")) \
    .filter(col("peso_coautoria") >= 2)  # Filtragem de colaborações recorrentes

total_pares = df_coautorias.count()
print(f"Total de pares de coautoria com colaboração recorrente (peso >= 2): {total_pares}")
display(df_coautorias.orderBy(col("peso_coautoria").desc()).limit(10))


## 2. Identificação dos Pesquisadores Centrais (Hubs)
Cruzamos os pares de coautoria com os metadados dos vértices para calcular o grau de centralidade de cada pesquisador e rankear os principais articuladores da rede.

In [ ]:
# Nomes dos autores a partir de gold_vertices
df_autores_info = df_vertices.filter(col("tipo_no") == "Autor") \
    .select(col("id_no").alias("id_autor"), col("nome_no").alias("nome_autor"))

# Contagem ponderada de conexões para cada ponta da aresta
df_g1 = df_coautorias.groupBy("autor_1").agg(count("*").alias("conexoes")).withColumnRenamed("autor_1", "id_autor")
df_g2 = df_coautorias.groupBy("autor_2").agg(count("*").alias("conexoes")).withColumnRenamed("autor_2", "id_autor")

df_ranking_hubs = df_g1.unionByName(df_g2) \
    .groupBy("id_autor").sum("conexoes") \
    .withColumnRenamed("sum(conexoes)", "grau_colaboracao") \
    .join(df_autores_info, on="id_autor", how="inner") \
    .orderBy(col("grau_colaboracao").desc())

print("--- TOP 10 PESQUISADORES MAIS CONECTADOS (HUBS DA REDE) ---")
display(df_ranking_hubs.limit(10))


## 3. Detecção de Comunidades Científicas (Modularity Clustering)
Construímos a representação estrutural em NetworkX e aplicamos o algoritmo de modularidade para particionar o grafo em grupos temáticos de pesquisa.

In [ ]:
# Coleta distribuída para processamento analítico de grafos
coautores_pd = df_coautorias.toPandas()
ranking_dict = df_ranking_hubs.toPandas().set_index("id_autor")["nome_autor"].to_dict()

G = nx.Graph()
for _, row in coautores_pd.iterrows():
    u, v, w = row["autor_1"], row["autor_2"], int(row["peso_coautoria"])
    G.add_edge(u, v, weight=w)

for node in G.nodes():
    G.nodes[node]["name"] = ranking_dict.get(node, "Autor")

# Seleciona o núcleo dos principais pesquisadores (Hubs) para renderização fluida na web
TOP_N_HUBS = 200
degrees_all = dict(G.degree(weight="weight"))
top_nodes = [n for n, _ in sorted(degrees_all.items(), key=lambda x: x[1], reverse=True)[:TOP_N_HUBS]]

H = G.subgraph(top_nodes).copy()
H.remove_nodes_from(list(nx.isolates(H)))

# Algoritmo de Louvain: ultrarrápido (executa em milissegundos)
try:
    communities = list(community.louvain_communities(H, seed=42))
except AttributeError:
    communities = list(community.greedy_modularity_communities(H))

print(f"Nós no Subgrafo: {H.number_of_nodes()} | Arestas: {H.number_of_edges()}")
print(f"Comunidades de pesquisa detectadas: {len(communities)}")


## 4. Renderização do Grafo Dinâmico Interativo na Web via Vis.js
Geramos a visualização com física de partículas ativa, permitindo arrastar nós, inspecionar clusters e aplicar zoom diretamente no Databricks com `displayHTML()`.

In [ ]:
palette = ["#38bdf8", "#f43f5e", "#10b981", "#fbbf24", "#a855f7", "#ec4899", "#14b8a6", "#f97316"]
degrees = dict(H.degree(weight="weight"))

node_comm = {}
for idx, comm in enumerate(communities):
    for n in comm:
        node_comm[n] = idx

vis_nodes = []
for node in H.nodes():
    name = ranking_dict.get(node, "Autor")
    deg = degrees[node]
    c_idx = node_comm.get(node, 0)
    color = palette[c_idx % len(palette)]
    vis_nodes.append({
        "id": node,
        "label": name,
        "value": deg,
        "title": f"<b>{name}</b><br>Grau Ponderado: {deg}<br>Cluster: {c_idx + 1}",
        "color": color,
        "font": {"color": "#f8fafc", "size": 12}
    })

vis_edges = []
for u, v, data in H.edges(data=True):
    w = data.get("weight", 1)
    vis_edges.append({
        "from": u,
        "to": v,
        "value": w,
        "title": f"{w} artigos conjuntos",
        "color": {"color": "#475569", "opacity": 0.4}
    })

html_content = f"""
<div style="width: 100%; height: 650px; background: #0b0f19; border-radius: 8px; border: 1px solid #1e293b; position: relative;">
    <div style="position: absolute; top: 12px; left: 16px; color: #f8fafc; font-family: sans-serif; z-index: 10;">
        <h3 style="margin: 0; font-size: 16px;">Rede de Coautoria em Inteligência Artificial</h3>
        <p style="margin: 4px 0 0 0; font-size: 11px; color: #94a3b8;">Camada Ouro (OpenAlex) • Clique e arraste os nós</p>
    </div>
    <div id="databricks-network" style="width: 100%; height: 100%;"></div>
</div>
<script type="text/javascript" src="https://unpkg.com/vis-network/standalone/umd/vis-network.min.js"></script>
<script type="text/javascript">
    const nodes = new vis.DataSet(" + json.dumps(vis_nodes) + ");\n    const edges = new vis.DataSet(" + json.dumps(vis_edges) + ");\n    const container = document.getElementById("databricks-network");
    const data = { nodes: nodes, edges: edges };
    const options = {
        nodes: { shape: "dot", scaling: { min: 10, max: 35 } },
        edges: { smooth: { type: "continuous" } },
        physics: {
            solver: "forceAtlas2Based",
            forceAtlas2Based: { gravitationalConstant: -60, springLength: 80, damping: 0.4 }
        }
    };
    new vis.Network(container, data, options);
</script>
"""

# Renderização nativa na célula do Databricks
displayHTML(html_content)


## 5. Exportação para Gephi (.gexf)
Gravamos o subgrafo enriquecido com graus e identificadores de comunidade em um arquivo `.gexf` no volume persistente do Unity Catalog.

In [ ]:
catalogo = spark.sql("SELECT current_catalog()").collect()[0][0]
esquema = spark.sql("SELECT current_database()").collect()[0][0]
volume_gexf = f"/Volumes/{catalogo}/{esquema}/bronze_volume/rede_coautoria_openalex.gexf"

for n in H.nodes():
    H.nodes[n]["label"] = ranking_dict.get(n, "Autor")
    H.nodes[n]["community"] = int(node_comm.get(n, 0))
    H.nodes[n]["degree"] = int(degrees[n])

try:
    nx.write_gexf(H, volume_gexf)
    print(f"Arquivo Gephi salvo com sucesso no Volume: {volume_gexf}")
except Exception as e:
    print(f"Nota de gravação: {e}")
